In [ ]:
# Настройка: импорты, базовый URL API, выходная папка, проверка что сервис живой

import io, os, time, wave
import requests
from PIL import Image

BASE = "http://localhost:7860/api"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Sanity: API alive?
requests.get(f"{BASE}/jobs", timeout=5).raise_for_status()
print("API up")

API up


In [ ]:
# Генерим тестовые входы в памяти: синий PNG 128×128 + 1-сек тишина WAV

img_buf = io.BytesIO()
Image.new("RGB", (128, 128), "#1565c0").save(img_buf, "PNG")
img_buf.seek(0)

aud_buf = io.BytesIO()
with wave.open(aud_buf, "wb") as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000)
    w.writeframes(b"\x00\x00" * 16000)  # 1 s silence
aud_buf.seek(0)

print(f"image: {len(img_buf.getvalue())} B  ·  audio: {len(aud_buf.getvalue())} B")

image: 362 B  ·  audio: 32044 B


In [ ]:
# Отправляем задачу: POST /api/jobs с файлами и mode=mock → получаем job_id

r = requests.post(
    f"{BASE}/jobs",
    files={
        "image": ("test.png", img_buf, "image/png"),
        "audio": ("test.wav", aud_buf, "audio/wav"),
    },
    data={"prompt": "smoke test", "mode": "mock"},
    timeout=30,
)
r.raise_for_status()
job_id = r.json()["job_id"]
print("submitted:", job_id)

submitted: 115f6f15d0a64e13ac7e43b64a6cb8e4


In [ ]:
# Опрашиваем GET /api/jobs/{id} раз в секунду пока статус не done или failed

while True:
    s = requests.get(f"{BASE}/jobs/{job_id}", timeout=5).json()
    print(f"{s['status']:>8}  {s['progress']:>5.1f}%   {s.get('message','')}")
    if s["status"] in ("done", "failed"):
        break
    time.sleep(1)

assert s["status"] == "done", f"job failed: {s.get('error')}"
print(f"\n✓ done in {s['elapsed_seconds']} s")

 running   40.0%   mock step 8/20
 running   70.0%   mock step 14/20
    done  100.0%   mock step 20/20

✓ done in 10.0 s


In [ ]:
# Скачиваем готовый mp4 через GET /api/jobs/{id}/result и сохраняем в outputs/

r = requests.get(f"{BASE}/jobs/{job_id}/result", timeout=30)
r.raise_for_status()
out_path = os.path.join(OUT_DIR, f"smoke_result_{job_id[:8]}.mp4")
with open(out_path, "wb") as f:
    f.write(r.content)
print(f"saved {len(r.content)} B → {out_path}")

saved 4175 B → outputs\smoke_result_115f6f15.mp4


In [ ]:
# Список всех задач через GET /api/jobs — для галереи и мониторинга

all_jobs = requests.get(f"{BASE}/jobs", timeout=5).json()
print(f"total jobs: {len(all_jobs)}\n")
for j in all_jobs[:5]:
    print(f"  {j['id'][:8]}  {j['status']:>8}  elapsed={j.get('elapsed_seconds')}s  mode={j['mode']}")

total jobs: 1

  115f6f15      done  elapsed=10.0s  mode=mock
